# 06 — ML Baselines & BH FDR Correction

Logistic regression, GBM, trivial anchors. Final pooled BH correction for all secondary cells.

In [1]:
import sys, os
from pathlib import Path
for _cand in ['.', '..']:
    if (Path(_cand)/'src').is_dir() and (Path(_cand)/'legacy').is_dir():
        os.chdir(_cand); break
sys.path.insert(0, 'src'); sys.path.insert(0, 'legacy/src')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.baselines import run_all_baselines
from src.grid import correct_grid
from src.report_io import save_fig, save_table, setup_style
from src.config import PANELS_2016

setup_style()
print('Setup complete.')

Setup complete.


## 1. Load AAPL panel for baseline comparison

In [2]:
aapl_path = PANELS_2016.get('AAPL')
if not aapl_path or not Path(aapl_path).exists():
    raise RuntimeError(f'AAPL 2016-2020 panel not found at {aapl_path}')

panel = pd.read_csv(aapl_path)
panel['stock'] = 'AAPL'
print(f'Loaded: {len(panel):,} rows')

Loaded: 6,252 rows


## 2. Run all baselines

In [3]:
baseline_rows = []
for h in [1, 5, 15]:
    try:
        res = run_all_baselines(panel, horizon=h)
        res['horizon'] = h
        baseline_rows.append(res)
    except Exception as e:
        print(f'  Baseline h={h} failed: {e}')

if baseline_rows:
    baseline_tbl = pd.concat(baseline_rows, ignore_index=True)
    display_cols = [c for c in ['model','horizon','accuracy','roc_auc','brier_score','cv_folds']
                    if c in baseline_tbl.columns]
    print(baseline_tbl[display_cols].to_string(index=False, float_format='{:.4f}'.format))
    save_table(
        baseline_tbl[display_cols],
        '06_baselines',
        caption='ML baseline performance (AAPL 2016-2020): logistic, GBM, always-long, random. 5-fold CV. Features: TF-IDF(headline) + ofi\_z + lm\_score.',
        label='tab:baselines',
    )


  Baseline results (ret_1m):
      model    n  cv_accuracy_mean  cv_accuracy_std  ic_proxy
   logistic 6252          0.508314         0.011205  0.180251
        gbm 6252          0.506714         0.011874  0.350654
always_long 6252          0.522553         0.000187       NaN
     random 6252          0.489922         0.008706       NaN



  Baseline results (ret_5m):
      model    n  cv_accuracy_mean  cv_accuracy_std  ic_proxy
   logistic 6252          0.501443         0.011602  0.173225
        gbm 6252          0.501442         0.011921  0.334043
always_long 6252          0.508317         0.000299       NaN
     random 6252          0.500316         0.015119       NaN



  Baseline results (ret_15m):
      model    n  cv_accuracy_mean  cv_accuracy_std  ic_proxy
   logistic 6252          0.490725         0.009451  0.159737
        gbm 6252          0.503522         0.012595  0.321807
always_long 6252          0.517115         0.000298       NaN
     random 6252          0.490878         0.018583       NaN
      model  horizon
   logistic        1
        gbm        1
always_long        1
     random        1
   logistic        5
        gbm        5
always_long        5
     random        5
   logistic       15
        gbm       15
always_long       15
     random       15
  Saved table  → results/tables/06_baselines.csv + results/tables/06_baselines.tex


## 3. Baselines figure

In [4]:
if baseline_rows:
    metric = 'roc_auc' if 'roc_auc' in baseline_tbl.columns else 'accuracy'
    models = baseline_tbl['model'].unique()
    horizons = sorted(baseline_tbl['horizon'].unique())

    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(horizons))
    width = 0.8 / max(len(models), 1)
    for i, model in enumerate(models):
        sub = baseline_tbl[baseline_tbl['model'] == model].sort_values('horizon')
        vals = sub[metric].values if metric in sub.columns else np.full(len(horizons), np.nan)
        offset = (i - len(models)/2 + 0.5) * width
        ax.bar(x + offset, vals, width=width*0.9, label=model)

    ax.axhline(0.5, color='black', lw=0.7, ls='--', alpha=0.5, label='chance')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{h}-min' for h in horizons])
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title('ML baselines: directional prediction')
    ax.legend(frameon=False, ncol=2)
    ax.grid(axis='y', alpha=0.2)
    save_fig(fig, '06_baselines')
    plt.close()

  Saved figure → results/figures/06_baselines.pdf


## 4. Pooled BH FDR correction (ALL secondary cells)

This cell must run LAST — all secondary cells from notebooks 01/02/04/05 must be appended first.

In [5]:
print('Running pooled BH FDR correction on all accumulated secondary cells:')
fdr_df = correct_grid(q=0.05)

if fdr_df.empty:
    print('No secondary cells yet — run notebooks 01-05 first, then re-run this cell.')
else:
    save_table(
        fdr_df,
        '06_secondary_grid_fdr',
        caption='All secondary IC tests with pooled BH FDR correction at $q=0.05$.',
        label='tab:secondary_fdr',
    )

Running pooled BH FDR correction on all accumulated secondary cells:

  BH FDR correction (q=0.05): 113 cells, 0 survive.
  No secondary cells survive FDR correction.
  Saved table  → results/tables/06_secondary_grid_fdr.csv + results/tables/06_secondary_grid_fdr.tex


## 5. Final summary printout

In [6]:
print('='*60)
print('FINAL REPORT SUMMARY')
print('='*60)

# Primary hypothesis result
grid_path = Path('results/tables/secondary_grid.csv')
if grid_path.exists():
    grid = pd.read_csv(grid_path)
    n_cells = len(grid)
    print(f'Secondary grid: {n_cells} total cells')

fdr_path = Path('results/tables/secondary_grid_fdr.csv')
if fdr_path.exists():
    fdr_df2 = pd.read_csv(fdr_path)
    survivors = fdr_df2[fdr_df2.get('reject_fdr', pd.Series(False))]
    print(f'BH survivors (q=0.05): {len(survivors)}/{len(fdr_df2)}')
    if not survivors.empty:
        print(survivors[['notebook','cell_id','stock','scorer','horizon','ic','p','p_adj']].to_string(index=False))

# LOB coverage
cost_path = Path('results/tables/05_aapl_net_ic.csv')
if cost_path.exists():
    ct = pd.read_csv(cost_path)
    if 'coverage_frac' in ct.columns:
        print(f'AAPL LOB coverage_frac: {ct["coverage_frac"].mean():.1%}')

# Figures saved
figs = list(Path('results/figures').glob('*.pdf')) if Path('results/figures').exists() else []
print(f'Figures saved: {len(figs)}')
for f in sorted(figs):
    print(f'  {f}')

# Tables saved
tbls = list(Path('results/tables').glob('*.csv')) if Path('results/tables').exists() else []
print(f'Tables saved:  {len(tbls)}')
for t in sorted(tbls):
    print(f'  {t}')

print('='*60)

FINAL REPORT SUMMARY
Secondary grid: 116 total cells
BH survivors (q=0.05): 0/113
AAPL LOB coverage_frac: 100.0%
Figures saved: 5
  results/figures/01_unconditional_ic.pdf
  results/figures/02_ic_decay.pdf
  results/figures/04_relevance_ic.pdf
  results/figures/05_gross_vs_net_ic.pdf
  results/figures/06_baselines.pdf
Tables saved:  13
  results/tables/01_unconditional_ic.csv
  results/tables/02_conditional_regression.csv
  results/tables/02_confirmed_conflicted_ic.csv
  results/tables/04_agent_vs_lexicon.csv
  results/tables/04_oos_ic.csv
  results/tables/04_relevance_ic.csv
  results/tables/05_aapl_net_ic.csv
  results/tables/05_net_ic_multi.csv
  results/tables/06_baselines.csv
  results/tables/06_secondary_grid_fdr.csv
  results/tables/relevance_sample_to_label.csv
  results/tables/secondary_grid.csv
  results/tables/secondary_grid_fdr.csv
